### Code flow
- Get csv file paths in volume
- Read dataframe from csv path
- Apply transformation:
  - Drop column: is_best_match
  - Add column: symbol (value: BTCUSDT)
  - Convert to Timestampt: aggregate_timestamp (to Millisecond)

### Config

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType, BooleanType

# catalog details
catalog = "smart-trading-dev"
schema = "00_volume"
volume = "trades_backfill_data"

file_name = "BTCUSDT-aggTrades-2026-02[1].csv"

# paths
path_catalog = "/Volumes/" + catalog
path_schema = path_catalog + "/" + schema
path_volume = path_schema + "/" + volume
path_file = path_volume + "/" + file_name

# schema
data_schema = StructType([
    StructField("aggregate_trade_id", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", DoubleType(), True),
    StructField("first_trade_id", LongType(), True),
    StructField("last_trade_id", LongType(), True),
    StructField("aggregate_timestamp", LongType(), True),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True)
])

In [0]:
from pyspark.sql.functions import col, lit, timestamp_millis

def escape_file_path(file_path):
    escaped_path = file_path.replace("[", "\\[").replace("]", "\\]").replace('dbfs:','')
    return escaped_path

def load_csv_to_dataframe(path, schema):
    df_backfill = (
        spark.read
        .format("csv")
        .schema(schema)
        .load(path)
    )
    return df_backfill

def transform_dataframe(dataframe):
    # drop column is_best_match
    dataframe = dataframe.drop("is_best_match")

    # add column symbol at index 1
    cols = dataframe.columns
    dataframe = dataframe.withColumn('symbol', lit('BTCUSDT'))
    cols = cols[:1] + ['symbol'] + cols[1:]
    dataframe = dataframe.select(*cols)

    # convert aggregate_timestamp from milliseconds to timestamp
    dataframe = dataframe.withColumn('aggregate_timestamp', (col('aggregate_timestamp') / 1000000).cast(TimestampType()))

    return dataframe

def write_dataframe_to_silver(dataframe):
    dataframe.writeTo("`smart-trading-dev`.`02_silver`.trades_silver").append()
    return

### Execution: load csv files to bronze

In [0]:
for file in dbutils.fs.ls(path_volume):
    file_path = file.path
    escaped_path = escape_file_path(file_path)
    df_backfill = load_csv_to_dataframe(escaped_path, data_schema)
    df_backfill = transform_dataframe(df_backfill)
    write_dataframe_to_silver(df_backfill)
    print(f"File {file_path} processed successfully.")
    

In [0]:
# %sql
# DELETE FROM `smart-trading-dev`.`02_silver`.trades_silver
# WHERE bronze_arrival_time IS NULL

### Testing

In [0]:
# from pyspark.sql.functions import lit, col


# file_path = dbutils.fs.ls(path_volume)[0].path
# escape_path = escape_file_path(file_path)
# df_backfill = load_csv_to_dataframe(escape_path, data_schema)
# df_backfill = transform_dataframe(df_backfill)

# display(df_backfill)